# <img align="left" src="./images/movie_camera.png"     style=" width:40px;  " > 练习实验：协同过滤推荐系统

在本练习中，你将实现协同过滤，构建电影推荐系统。

# <img align="left" src="./images/film_reel.png"     style=" width:40px;  " > 大纲
- [ 1 - 记号](#1)
- [ 2 - 推荐系统](#2)
- [ 3 - 电影评分数据集](#3)
- [ 4 - 协同过滤学习算法](#4)
  - [ 4.1 协同过滤代价函数](#4.1)
    - [ 练习 1](#ex01)
- [ 5 - 学习电影推荐](#5)
- [ 6 - 推荐](#6)
- [ 7 - 恭喜！](#7)



## 软件包 <img align="left" src="./images/film_strip_vertical.png"     style=" width:40px;   " >
我们将使用现在已经熟悉的 NumPy 和 TensorFlow 软件包。

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from recsys_utils import *

<a name="1"></a>
## 1 - 记号

|通用 <br />  记号  | 描述| Python（如适用） |
|:-------------|:------------------------------------------------------------||
| $r(i,j)$     | 标量；如果用户 j 评价了游戏 i，则为 1，否则为 0             ||
| $y(i,j)$     | 标量；用户 j 对游戏 i 给出的评分（如果定义了 r(i,j) = 1） ||
|$\mathbf{w}^{(j)}$ | 向量；用户 j 的参数 ||
|$b^{(j)}$     | 标量；用户 j 的参数 ||
| $\mathbf{x}^{(i)}$ | 向量；电影 i 的特征评分        ||
| $n_u$        | 用户数量 |num_users|
| $n_m$        | 电影数量 | num_movies |
| $n$          | 特征数量 | num_features                    |
| $\mathbf{X}$ | 向量矩阵 $\mathbf{x}^{(i)}$         | X |
| $\mathbf{W}$ | 向量矩阵 $\mathbf{w}^{(j)}$         | W |
| $\mathbf{b}$ | 偏置参数向量 $b^{(j)}$ | b |
| $\mathbf{R}$ | 元素矩阵 $r(i,j)$                    | R |


<a name="2"></a>
## 2 - 推荐系统 <img align="left" src="./images/film_rating.png" style=" width:40px;  " >
在本实验中，你将实现协同过滤学习算法，并将其应用于电影评分数据集。
协同过滤推荐系统的目标是生成两类向量：为每位用户生成一个体现其电影偏好的“参数向量”；为每部电影生成一个相同大小、体现该电影某些描述信息的特征向量。两个向量的点积再加上偏置项，应能估计该用户可能给这部电影的评分。

下图详细说明了如何学习这些向量。

<figure>
   <img src="./images/ColabFilterLearn.PNG"  style="width:740px;height:250px;" >
</figure>

现有评分以图示矩阵形式提供。$Y$ 包含评分，评分从 0.5 到 5（含 0.5 和 5），步长为 0.5；如果电影未被评分，则值为 0。$R$ 在电影已被评分的位置取值为 1。电影按行排列，用户按列排列。每位用户都有一个参数向量 $w^{user}$ 和一个偏置；每部电影都有一个特征向量 $x^{movie}$。以现有的用户/电影评分作为训练数据，同时学习这些向量。上面显示了一个训练样本：$\mathbf{w}^{(1)} \cdot \mathbf{x}^{(1)} + b^{(1)} = 4$。值得注意的是，特征向量 $x^{movie}$ 必须满足所有用户，而用户向量 $w^{user}$ 必须满足所有电影。这正是该方法名称的由来——所有用户协同生成评分集。

<figure>
   <img src="./images/ColabFilterUse.PNG"  style="width:640px;height:250px;" >
</figure>

学到特征向量和参数后，可以用它们预测用户可能会如何评价一部尚未评分的电影。上图展示了这一过程。公式示例预测了用户 1 对电影 0 的评分。


在本练习中，你将实现函数 `cofiCostFunc`，用于计算协同过滤目标函数。实现目标函数后，你将使用 TensorFlow 自定义训练循环来学习协同过滤的参数。第一步是详细说明实验中使用的数据集和数据结构。

<a name="3"></a>
## 3 - 电影评分数据集 <img align="left" src="./images/film_rating.png"     style=" width:40px;  " >
该数据集派生自 [MovieLens “ml-latest-small”](https://grouplens.org/datasets/movielens/latest/) 数据集。  
[F. Maxwell Harper 和 Joseph A. Konstan。2015。《MovieLens 数据集：历史与背景》。ACM Transactions on Interactive Intelligent Systems (TiiS) 5, 4: 19:1–19:19。<https://doi.org/10.1145/2827872>]

原始数据集包含由 600 位用户评分的 9000 部电影。为了聚焦于 2000 年以来的电影，数据集的规模已经缩小。该数据集中的评分范围为 0.5 到 5，增量为 0.5。缩减后的数据集包含 $n_u = 443$ 位用户和 $n_m= 4778$ 部电影。

下面，你将把电影数据集加载到变量 $Y$ 和 $R$ 中。

矩阵 $Y$（一个 $n_m \times n_u$ 矩阵）存储评分 $y^{(i,j)}$。矩阵 $R$ 是一个二值指示矩阵；如果用户 $j$ 对电影 $i$ 进行了评分，则 $R(i,j) = 1$，否则 $R(i,j)=0$。

在本练习的这一部分中，你还将使用矩阵 $\mathbf{X}$、$\mathbf{W}$ 和 $\mathbf{b}$：

$$\mathbf{X} = 
\begin{bmatrix}
--- (\mathbf{x}^{(0)})^T --- \\
--- (\mathbf{x}^{(1)})^T --- \\
\vdots \\
--- (\mathbf{x}^{(n_m-1)})^T --- \\
\end{bmatrix} , \quad
\mathbf{W} = 
\begin{bmatrix}
--- (\mathbf{w}^{(0)})^T --- \\
--- (\mathbf{w}^{(1)})^T --- \\
\vdots \\
--- (\mathbf{w}^{(n_u-1)})^T --- \\
\end{bmatrix},\quad
\mathbf{ b} = 
\begin{bmatrix}
 b^{(0)}  \\
 b^{(1)} \\
\vdots \\
b^{(n_u-1)} \\
\end{bmatrix}\quad
$$ 

$\mathbf{X}$ 的第 $i$ 行对应第 $i$ 部电影的特征向量 $x^{(i)}$，而 $\mathbf{W}$ 的第 $j$ 行对应第 $j$ 位用户的一个参数向量 $\mathbf{w}^{(j)}$。$x^{(i)}$ 和 $\mathbf{w}^{(j)}$ 都是 $n$ 维向量。在本练习中，你将使用 $n=10$，因此 $\mathbf{x}^{(i)}$ 和 $\mathbf{w}^{(j)}$ 都有 10 个元素。
相应地，$\mathbf{X}$ 是一个 $n_m \times 10$ 矩阵，而 $\mathbf{W}$ 是一个 $n_u \times 10$ 矩阵。

我们先加载电影评分数据集，以了解数据结构。
我们会将电影数据集加载到 $Y$ 和 $R$ 中。  
我们还会将预先计算好的值加载到 $\mathbf{X}$、$\mathbf{W}$ 和 $\mathbf{b}$ 中。这些值将在实验后面学习得到，但这里我们将使用预计算值来构建代价模型。

In [ ]:
#Load data
X, W, b, num_movies, num_features, num_users = load_precalc_params_small()
Y, R = load_ratings_small()

print("Y", Y.shape, "R", R.shape)
print("X", X.shape)
print("W", W.shape)
print("b", b.shape)
print("num_features", num_features)
print("num_movies",   num_movies)
print("num_users",    num_users)

In [ ]:
#  From the matrix, we can compute statistics like average rating.
tsmean =  np.mean(Y[0, R[0, :].astype(bool)])
print(f"Average rating for movie 1 : {tsmean:0.3f} / 5" )

<a name="4"></a>
## 4 - 协同过滤学习算法 <img align="left" src="./images/film_filter.png"     style=" width:40px;  " >

现在，你将开始实现协同过滤学习算法。你将从实现目标函数开始。

在电影推荐场景中，协同过滤算法考虑一组 $n$ 维参数向量 $\mathbf{x}^{(0)},...,\mathbf{x}^{(n_m-1)}$、$\mathbf{w}^{(0)},...,\mathbf{w}^{(n_u-1)}$ 和 $b^{(0)},...,b^{(n_u-1)}$，其中模型将用户 $j$ 对电影 $i$ 的评分预测为 $y^{(i,j)} = \mathbf{w}^{(j)}\cdot \mathbf{x}^{(i)} + b^{(i)}$。给定一个由部分用户对部分电影所做评分组成的数据集，你希望学习参数向量 $\mathbf{x}^{(0)},...,\mathbf{x}^{(n_m-1)},
\mathbf{w}^{(0)},...,\mathbf{w}^{(n_u-1)}$  and $b^{(0)},...,b^{(n_u-1)}$，以获得最佳拟合（使平方误差最小）。

你将完成 cofiCostFunc 中的代码，以计算协同过滤的代价函数。


<a name="4.1"></a>
### 4.1 协同过滤代价函数

协同过滤代价函数为
$$J({\mathbf{x}^{(0)},...,\mathbf{x}^{(n_m-1)},\mathbf{w}^{(0)},b^{(0)},...,\mathbf{w}^{(n_u-1)},b^{(n_u-1)}})= \frac{1}{2}\sum_{(i,j):r(i,j)=1}(\mathbf{w}^{(j)} \cdot \mathbf{x}^{(i)} + b^{(j)} - y^{(i,j)})^2
+\underbrace{
\frac{\lambda}{2}
\sum_{j=0}^{n_u-1}\sum_{k=0}^{n-1}(\mathbf{w}^{(j)}_k)^2
+ \frac{\lambda}{2}\sum_{i=0}^{n_m-1}\sum_{k=0}^{n-1}(\mathbf{x}_k^{(i)})^2
}_{regularization}
\tag{1}$$
(1) 中的第一个求和表示“对 $r(i,j)$ 等于 $1$ 的所有 $i$、$j$ 求和”，可以写为：

$$
= \frac{1}{2}\sum_{j=0}^{n_u-1} \sum_{i=0}^{n_m-1}r(i,j)*(\mathbf{w}^{(j)} \cdot \mathbf{x}^{(i)} + b^{(j)} - y^{(i,j)})^2
+\text{regularization}
$$

现在应编写 cofiCostFunc（协同过滤代价函数）以返回此代价。

<a name="ex01"></a>
### 练习 1

**for 循环实现：**  
首先使用 for 循环实现代价函数。
可以分两步开发代价函数。第一步先实现不含正则化的代价函数。下面提供了一个不含正则化的测试用例，用于测试你的实现。确认其正常工作后，再添加正则化并运行包含正则化的测试。请注意，只有当 $R(i,j) = 1$ 时，才应累加用户 $j$ 和电影 $i$ 的代价。

In [ ]:
# GRADED FUNCTION: cofi_cost_func
# UNQ_C1

def cofi_cost_func(X, W, b, Y, R, lambda_):
    """
    Returns the cost for the content-based filtering
    Args:
      X (ndarray (num_movies,num_features)): matrix of item features
      W (ndarray (num_users,num_features)) : matrix of user parameters
      b (ndarray (1, num_users)            : vector of user parameters
      Y (ndarray (num_movies,num_users)    : matrix of user ratings of movies
      R (ndarray (num_movies,num_users)    : matrix, where R(i, j) = 1 if the i-th movies was rated by the j-th user
      lambda_ (float): regularization parameter
    Returns:
      J (float) : Cost
    """
    nm, nu = Y.shape
    J = 0
    ### START CODE HERE ###  
    
        
        
        
            
            
            
            
    
    
    ### END CODE HERE ### 

    return J

In [ ]:
# Public tests
from public_tests import *
test_cofi_cost_func(cofi_cost_func)

<details>
  <summary><font size="3" color="darkgreen"><b>点击查看提示</b></font></summary>
    可以使用两个 for 循环来组织代码，与 (1) 中的求和类似。  
    首先实现不含正则化的代码。  
    请注意，(1) 中的一些元素是向量。请使用 np.dot()，也可以使用 np.square()。
    请特别留意哪些元素按 i 索引，哪些元素按 j 索引。不要忘记除以 2。
    
```python     
    ### START CODE HERE ###  
    for j in range(nu):
        
        
        for i in range(nm):
            
            
    ### END CODE HERE ### 
```
<details>
    <summary><font size="2" color="darkblue"><b> 点击查看更多提示</b></font></summary>
        
    下面是更多详细信息。以下代码会先从矩阵中取出每个元素，然后再使用它。
    也可以直接引用矩阵。  
    此代码不包含正则化。
    
```python 
    nm,nu = Y.shape
    J = 0
    ### START CODE HERE ###  
    for j in range(nu):
        w = W[j,:]
        b_j = b[0,j]
        for i in range(nm):
            x = 
            y = 
            r =
            J += 
    J = J/2
    ### END CODE HERE ### 

```
    
<details>
    <summary><font size="2" color="darkblue"><b>最后手段（完整的非正则化实现）</b></font></summary>
    
```python 
    nm,nu = Y.shape
    J = 0
    ### START CODE HERE ###  
    for j in range(nu):
        w = W[j,:]
        b_j = b[0,j]
        for i in range(nm):
            x = X[i,:]
            y = Y[i,j]
            r = R[i,j]
            J += np.square(r * (np.dot(w,x) + b_j - y ) )
    J = J/2
    ### END CODE HERE ### 
```
    
<details>
    <summary><font size="2" color="darkblue"><b>正则化</b></font></summary>
     正则化只需对 W 数组和 X 数组中的每个元素求平方，然后将所有平方后的元素求和。
     可以使用 np.square() 和 np.sum()。

<details>
    <summary><font size="2" color="darkblue"><b>正则化详细信息</b></font></summary>
    
```python 
    J += lambda_* (np.sum(np.square(W)) + np.sum(np.square(X)))
```
    
</details>
</details>
</details>
</details>

    

In [ ]:
# Reduce the data set size so that this runs faster
num_users_r = 4
num_movies_r = 5 
num_features_r = 3

X_r = X[:num_movies_r, :num_features_r]
W_r = W[:num_users_r,  :num_features_r]
b_r = b[0, :num_users_r].reshape(1,-1)
Y_r = Y[:num_movies_r, :num_users_r]
R_r = R[:num_movies_r, :num_users_r]

# Evaluate cost function
J = cofi_cost_func(X_r, W_r, b_r, Y_r, R_r, 0);
print(f"Cost: {J:0.2f}")

**预期输出（lambda = 0）**：  
$13.67$。

In [ ]:
# Evaluate cost function with regularization 
J = cofi_cost_func(X_r, W_r, b_r, Y_r, R_r, 1.5);
print(f"Cost (with regularization): {J:0.2f}")

**预期输出**：

28.09

**向量化实现**

创建一个向量化实现来计算 $J$ 非常重要，因为后续优化期间会多次调用它。这里使用的线性代数并非本系列课程的重点，因此直接提供了实现。如果你是线性代数专家，可以不参考下方代码，自行创建版本。

运行下面的代码，验证它能否生成与非向量化版本相同的结果。

In [ ]:
def cofi_cost_func_v(X, W, b, Y, R, lambda_):
    """
    Returns the cost for the content-based filtering
    Vectorized for speed. Uses tensorflow operations to be compatible with custom training loop.
    Args:
      X (ndarray (num_movies,num_features)): matrix of item features
      W (ndarray (num_users,num_features)) : matrix of user parameters
      b (ndarray (1, num_users)            : vector of user parameters
      Y (ndarray (num_movies,num_users)    : matrix of user ratings of movies
      R (ndarray (num_movies,num_users)    : matrix, where R(i, j) = 1 if the i-th movies was rated by the j-th user
      lambda_ (float): regularization parameter
    Returns:
      J (float) : Cost
    """
    j = (tf.linalg.matmul(X, tf.transpose(W)) + b - Y)*R
    J = 0.5 * tf.reduce_sum(j**2) + (lambda_/2) * (tf.reduce_sum(X**2) + tf.reduce_sum(W**2))
    return J

In [ ]:
# Evaluate cost function
J = cofi_cost_func_v(X_r, W_r, b_r, Y_r, R_r, 0);
print(f"Cost: {J:0.2f}")

# Evaluate cost function with regularization 
J = cofi_cost_func_v(X_r, W_r, b_r, Y_r, R_r, 1.5);
print(f"Cost (with regularization): {J:0.2f}")

**预期输出**：  
代价：13.67  
代价（带正则化）：28.09

<a name="5"></a>
## 5 - 学习电影推荐 <img align="left" src="./images/film_man_action.png" style=" width:40px;  " >
------------------------------

完成协同过滤代价函数的实现后，就可以开始训练算法，为自己生成电影推荐。

在下面的单元格中，你可以输入自己选择的电影。之后算法会为你生成推荐！我们已根据自己的偏好填写了一些值，但在使用这些选择成功运行后，你应该修改它们，使其与你的喜好相符。
数据集中所有电影的列表位于[电影列表](data/small_movie_list.csv)文件中。

In [ ]:
movieList, movieList_df = load_Movie_List_pd()

my_ratings = np.zeros(num_movies)          #  Initialize my ratings

# Check the file small_movie_list.csv for id of each movie in our dataset
# For example, Toy Story 3 (2010) has ID 2700, so to rate it "5", you can set
my_ratings[2700] = 5 

#Or suppose you did not enjoy Persuasion (2007), you can set
my_ratings[2609] = 2;

# We have selected a few movies we liked / did not like and the ratings we
# gave are as follows:
my_ratings[929]  = 5   # Lord of the Rings: The Return of the King, The
my_ratings[246]  = 5   # Shrek (2001)
my_ratings[2716] = 3   # Inception
my_ratings[1150] = 5   # Incredibles, The (2004)
my_ratings[382]  = 2   # Amelie (Fabuleux destin d'Amélie Poulain, Le)
my_ratings[366]  = 5   # Harry Potter and the Sorcerer's Stone (a.k.a. Harry Potter and the Philosopher's Stone) (2001)
my_ratings[622]  = 5   # Harry Potter and the Chamber of Secrets (2002)
my_ratings[988]  = 3   # Eternal Sunshine of the Spotless Mind (2004)
my_ratings[2925] = 1   # Louis Theroux: Law & Disorder (2008)
my_ratings[2937] = 1   # Nothing to Declare (Rien à déclarer)
my_ratings[793]  = 5   # Pirates of the Caribbean: The Curse of the Black Pearl (2003)
my_rated = [i for i in range(len(my_ratings)) if my_ratings[i] > 0]

print('\nNew user ratings:\n')
for i in range(len(my_ratings)):
    if my_ratings[i] > 0 :
        print(f'Rated {my_ratings[i]} for  {movieList_df.loc[i,"title"]}');

现在，让我们将这些评分添加到 $Y$ 和 $R$，并对评分进行归一化。

In [ ]:
# Reload ratings and add new ratings
Y, R = load_ratings_small()
Y    = np.c_[my_ratings, Y]
R    = np.c_[(my_ratings != 0).astype(int), R]

# Normalize the Dataset
Ynorm, Ymean = normalizeRatings(Y, R)

让我们准备训练模型。初始化参数并选择 Adam 优化器。

In [ ]:
#  Useful Values
num_movies, num_users = Y.shape
num_features = 100

# Set Initial Parameters (W, X), use tf.Variable to track these variables
tf.random.set_seed(1234) # for consistent results
W = tf.Variable(tf.random.normal((num_users,  num_features),dtype=tf.float64),  name='W')
X = tf.Variable(tf.random.normal((num_movies, num_features),dtype=tf.float64),  name='X')
b = tf.Variable(tf.random.normal((1,          num_users),   dtype=tf.float64),  name='b')

# Instantiate an optimizer.
optimizer = keras.optimizers.Adam(learning_rate=1e-1)

现在训练协同过滤模型。这将学习参数 $\mathbf{X}$、$\mathbf{W}$ 和 $\mathbf{b}$。

同时学习 $w$、$b$ 和 $x$ 所涉及的操作，不属于 TensorFlow 神经网络软件包提供的典型“层”。因此，课程 2 中使用的 Model、Compile()、Fit()、Predict() 流程无法直接应用。我们可以改用自定义训练循环。

回顾一下早期实验中的梯度下降步骤：
- 重复直至收敛：
    - 计算前向传播
    - 计算损失相对于参数的导数
    - 使用学习率和计算出的导数更新参数
    
TensorFlow 拥有自动计算导数的强大功能，如下所示。在 `tf.GradientTape()` 代码段中，会跟踪对 TensorFlow 变量执行的操作。稍后调用 `tape.gradient()` 时，它会返回损失相对于被跟踪变量的梯度。然后，可以使用优化器将这些梯度应用到参数。
这里只是对 TensorFlow 及其他机器学习框架中一个实用功能的简要介绍。如需更多信息，可以研究所用框架中的“自定义训练循环”。
    

In [ ]:
iterations = 200
lambda_ = 1
for iter in range(iterations):
    # Use TensorFlow’s GradientTape
    # to record the operations used to compute the cost 
    with tf.GradientTape() as tape:

        # Compute the cost (forward pass included in cost)
        cost_value = cofi_cost_func_v(X, W, b, Ynorm, R, lambda_)

    # Use the gradient tape to automatically retrieve
    # the gradients of the trainable variables with respect to the loss
    grads = tape.gradient( cost_value, [X,W,b] )

    # Run one step of gradient descent by updating
    # the value of the variables to minimize the loss.
    optimizer.apply_gradients( zip(grads, [X,W,b]) )

    # Log periodically.
    if iter % 20 == 0:
        print(f"Training loss at iteration {iter}: {cost_value:0.1f}")

<a name="6"></a>
## 6 - 推荐
下面，我们计算所有电影和用户的评分，并显示推荐的电影。这些推荐基于上面作为 `my_ratings[]` 输入的电影和评分。要预测用户 $j$ 对电影 $i$ 的评分，需要计算 $\mathbf{w}^{(j)} \cdot \mathbf{x}^{(i)} + b^{(j)}$。可以使用矩阵乘法计算所有评分。

In [ ]:
# Make a prediction using trained weights and biases
p = np.matmul(X.numpy(), np.transpose(W.numpy())) + b.numpy()

#restore the mean
pm = p + Ymean

my_predictions = pm[:,0]

# sort predictions
ix = tf.argsort(my_predictions, direction='DESCENDING')

for i in range(17):
    j = ix[i]
    if j not in my_rated:
        print(f'Predicting rating {my_predictions[j]:0.2f} for movie {movieList[j]}')

print('\n\nOriginal vs Predicted ratings:\n')
for i in range(len(my_ratings)):
    if my_ratings[i] > 0:
        print(f'Original {my_ratings[i]}, Predicted {my_predictions[i]:0.2f} for {movieList[i]}')

在实践中，可以利用额外信息来增强预测。在上面，前几百部电影的预测评分处于一个很小的范围内。我们可以在上述基础上进一步筛选：从排名靠前的电影中，选择平均评分较高且评分次数超过 20 次的电影。本节使用 [Pandas](https://pandas.pydata.org/) DataFrame，它提供了许多便捷的排序功能。

In [ ]:
filter=(movieList_df["number of ratings"] > 20)
movieList_df["pred"] = my_predictions
movieList_df = movieList_df.reindex(columns=["pred", "mean rating", "number of ratings", "title"])
movieList_df.loc[ix[:300]].loc[filter].sort_values("mean rating", ascending=False)

<a name="7"></a>
## 7 - 恭喜！<img align="left" src="./images/film_award.png"     style=" width:40px;  " >
你已经实现了一个实用的推荐系统！